A shallow neural network from scratch, working on the dataset "Pima Indians Diabetes", with pre-processing the data and evaluation of the model.

## Importing libraries

In [7]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

## Dataset
I used the simple dataset "Pima Indians Diabetes". This dataset contains the information of 768 native American women from the Pima tribe, reviewing the factors related to type 2 diabetes.
This dataset contains age, weight, height, family history of diabetes, blood pressure, blood glucose level, and other factors.

| Column | Description |
| ------ | ----------- |
| `Pregnancies` | the number of pregnancies |
| `Glucose` | blood glucose level (`mg/dL`) |
| `BloodPressure` | systolic blood pressure (`mmHg`) |
| `SkinThickness` | the thickness of skin (`mm`) |
| `Insulin` | blood insulin level (`μU/mL`) |
| `BMI` | body-mass-index (`kg/m^2`) |
| `DiabetesPedigreeFunction` | a function showing family diabetes history |
| `Age` | age (years) |
| `Outcome` | having diabetes (`1`) or not having it (`0`) |

In [2]:
train_data = pd.read_csv('data/diabetes_train.csv')
train_data.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [3]:
test_data = pd.read_csv('data/diabetes_test.csv')
test_data.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
0,6,98,58,33,190,34.0,0.430,43
1,9,154,78,30,100,30.9,0.164,45
2,6,165,68,26,168,33.6,0.631,49
3,1,99,58,10,0,25.4,0.551,21
4,10,68,106,23,49,35.5,0.285,47


## Preprocessing and Feature Engineering

In [4]:
train_data_outcome = train_data['Outcome']
train_data = train_data.drop('Outcome', axis=1)

train_data.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
0,6,148,72,35,0,33.6,0.627,50
1,1,85,66,29,0,26.6,0.351,31
2,8,183,64,0,0,23.3,0.672,32
3,1,89,66,23,94,28.1,0.167,21
4,0,137,40,35,168,43.1,2.288,33


### Normalization

$$
Z = \frac{x_{i} - \overline{x}}{\sigma}
$$

In [5]:
for column in train_data.columns:
    mean = train_data[column].mean()
    std = train_data[column].std()
    train_data[column]=(train_data[column]- mean)/std
    test_data[column]=(test_data[column] - mean)/std
    
train_data.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
0,0.649833,0.854539,0.166518,0.900880,-0.687695,0.222281,0.438405,1.443781
1,-0.835754,-1.096441,-0.140758,0.526362,-0.687695,-0.672046,-0.370035,-0.178571
2,1.244068,1.938416,-0.243184,-1.283807,-0.687695,-1.093658,0.570216,-0.093184
3,-0.835754,-0.972569,-0.140758,0.151844,0.123855,-0.480405,-0.908995,-1.032441
4,-1.132872,0.513891,-1.472290,0.900880,0.762734,1.436011,5.303692,-0.007797


### Adding bias to dataframe

In [6]:
train_bias = [1] * len(train_data)
train_data = train_data.assign(Bias=train_bias)

test_bias = [1] * len(test_data)
test_data = test_data.assign(Bias=test_bias)

test_data.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Bias
0,0.649833,-0.693858,-0.550460,0.776041,0.952672,0.273386,-0.138634,0.846073,1
1,1.541186,1.040346,0.473794,0.588782,0.175656,-0.122674,-0.917783,1.016847,1
2,0.649833,1.380993,-0.038333,0.339103,0.762734,0.222281,0.450122,1.358395,1
3,-0.835754,-0.662890,-0.550460,-0.659611,-0.687695,-0.825359,0.215791,-1.032441,1
4,1.838303,-1.622896,1.907751,0.151844,-0.264653,0.465027,-0.563358,1.187621,1


### Splitting the dataframe for train and test

In [8]:
X_train, X_validation, y_train, y_validation = train_test_split(train_data, train_data_outcome, test_size=0.2) 

X_train = np.array(X_train).T
X_validation = np.array(X_validation).T
y_train = np.array(y_train).T
y_validation = np.array(y_validation).T
test_data_numpy = np.array(test_data).T

Ensuring of the correctness:

In [9]:
print(f'X_train.shape:{X_train.shape}, y_train.shape:{y_train.shape}')
print(f'X_validation.shape:{X_validation.shape}, y_validation.shape:{y_validation.shape}')
print(f'test_data_numpy.shape:{test_data_numpy.shape}')

X_train.shape:(9, 534), y_train.shape:(534,)
X_validation.shape:(9, 134), y_validation.shape:(134,)
test_data_numpy.shape:(9, 100)


The above cell should have this output:

```
X_train.shape:(9, 534), y_train.shape:(534,)
X_validation.shape:(9, 134), y_validation.shape:(134,)
test_data_numpy.shape:(9, 100)
```

## Model

A shallow neural network with one hidden layer (with `1000` neurons) and the rectified linear unit activation function (`ReLu`). The activation function of the output later is sigmoid. 

```py
sigmoid_Z = 1 / (1 + np.exp(-Z))
ReLu_Z = np.maximum(0, Z)
```

- Sigmoid function:

| Sigmoid Function | Derivative of Sigmoid Function |
| ---------------- | ------------------------------ |
| $f(z) = \frac{1}{1 + e^{-z}}$ | $f'(z) = f(z) (1 - f(z))$ |

- ReLu function:

| ReLu Function | Derivative of ReLu Function |
| ------------- | --------------------------- |
| $$f(z) = \begin{cases} 0 & \text{if } z < 0 \\ z & \text{if } z \geq 0\end{cases}$$ | $$f'(z) = \begin{cases} 0 & \text{if } z < 0 \\ 1 & \text{if } z \geq 0\end{cases}$$ |

### `__init__` method

In this method we initialize the weights of the hidden layer and output layer (`w1` and `w2`) randomly, with the mean $0$ and the standard deviation $0.01$.

### `predict` method

The method `predict(self, inputs)` gets the inputs, and returns the output of both layers (`A_1` and `A_2`).

The formulas for these actions:

$$Z^{[1]}=W^{[1]}.X$$
$$A^{[1]}= \operatorname{ReLU}(Z^{[1]})$$
$$Z^{[2]}=W^{[2]}A^{[1]}$$
$$A^{[2]}=\sigma(Z^{[2]})=\frac{1}{1+e^{-Z^{[2]}}}=Y_{pred}$$

### `update_weights_for_one_epoch` method

The method `update_weights_for_one_epoch(self, inputs, outputs, learning_rate)` updates the weights of the network for one `epoch`. The variable `learning_rate` is $\alpha$.

This is how `w2` is updates:

$$W^{[2]} = W^{[2]} + \Delta W^{[2]}$$
$$\Delta W^{[2]} = - \alpha \frac{\partial cost}{\partial W^{[2]}}$$
$$\frac{\partial cost}{\partial W^{[2]}} = (\frac{-2}{n}(Y_{true}-A^{[2]})\odot A^{[2]}\odot (1-A^{[2]}))\bullet A^{[1]T}$$
$$W^{[2]}=W^{[2]}+(\frac{2 \alpha}{n}(Y_{true}-A^{[2]})\odot A^{[2]}\odot (1-A^{[2]}))\bullet A^{[1]T}$$

And this is how `w1` is updated:

$$W^{[1]} = W^{[1]} + \Delta W^{[1]}$$
$$\Delta W^{[1]} = - \alpha \frac{\partial cost}{\partial W^{[1]}}$$

$$\frac{\partial cost}{\partial W^{[1]}} = (((\frac{-2}{n}(Y_{true}-A^{[2]})\odot A^{[2]}\odot (1-A^{[2]}))^T\bullet W^{[2]})^T\odot \frac{\partial A^{[1]}}{\partial Z^{[1]}}) \bullet X^T$$

$$W^{[1]}=W^{[1]}+(((\frac{2 \alpha}{n}(Y_{true}-A^{[2]})\odot A^{[2]}\odot (1-A^{[2]}))^T\bullet W^{[2]})^T\odot \frac{\partial A^{[1]}}{\partial Z^{[1]}}) \bullet X^T$$

The value of $\frac{\partial A^{[1]}}{\partial Z^{[1]}}$ is calculated using the code snippet below:

```py
relu_gradient = np.where(A_1 > 0, 1, 0)
```

And a part of $\Delta W^{[1]}$ is calculated in $\Delta W^{[2]}$, so we kept it as variable to avoid additional calculations.

### `fit` method

The mthod `fit(self, inputs, outputs, learning_rate, epochs=64)` updates the weights of the network to the number of `epochs`.

## Model Implementation

The Model class is implemented in [`model.py`](model.py).

In [11]:
from model import Model

model = Model()

## Training and Evaluation of the model

In [12]:
def evaluation(model, inputs, outputs):
    _, A_2 = model.predict(inputs)
    prediction = (A_2 > 0.5).astype(int)
    outputs_binary = (outputs > 0).astype(int).reshape(1, -1)
    return np.mean(prediction == outputs_binary) * 100

Training the model:

In [13]:
model.fit(inputs=X_train, outputs=y_train, learning_rate = 0.1, epochs = 200)

Epoch 0, Loss: 0.25018351620239143
Epoch 10, Loss: 0.24764920695058798
Epoch 20, Loss: 0.24505415895067675
Epoch 30, Loss: 0.24222181323558625
Epoch 40, Loss: 0.23900346463538472
Epoch 50, Loss: 0.23527801817473107
Epoch 60, Loss: 0.23096608626346551
Epoch 70, Loss: 0.22605006849783432
Epoch 80, Loss: 0.22060883946487755
Epoch 90, Loss: 0.21479970588081718
Epoch 100, Loss: 0.20886486228376464
Epoch 110, Loss: 0.2030714844779177
Epoch 120, Loss: 0.19764229354509777
Epoch 130, Loss: 0.19272942221234776
Epoch 140, Loss: 0.18838783379979127
Epoch 150, Loss: 0.18460814133504755
Epoch 160, Loss: 0.1813404632330003
Epoch 170, Loss: 0.17851375672028572
Epoch 180, Loss: 0.17605499579440598
Epoch 190, Loss: 0.17389703400962966


In [14]:
print(f"you model accuracy on given set: {evaluation(model, X_train, y_train)}%")

you model accuracy on given set: 76.59176029962546%


## Predictions of Test Dataset

In [15]:
_ , output = model.predict(test_data_numpy)
prediction = (output > 0.5).flatten()